# Aegis Momentum - MACD Crossover Backtest (Nifty 50)
This notebook backtests the MACD Crossover Momentum strategy across all Nifty 50 stocks using daily historical data from the Zerodha Kite Connect API.

### Strategy Rules:
1. **Buy Entry**: MACD Line crosses above the Signal Line (Upward Crossover).
2. **Sell Exit**: MACD Line crosses below the Signal Line (Downward Crossover) OR 20 trading days hold limit reached.
3. **Uptrend Filter**: Price close must be above its 200-day SMA.
4. **Price Momentum Filter**: Price close must have a positive 3-day return (confirming price change is building up).

In [2]:
import os
import pandas as pd
import numpy as np
import requests
from pathlib import Path

# Load .env file configurations
def load_env():
    env_path = Path('.env')
    if env_path.exists():
        with open(env_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                if '=' in line:
                    k, v = line.split('=', 1)
                    os.environ[k.strip()] = v.strip().strip("'").strip('"')
                    os.environ[k.strip().upper()] = v.strip().strip("'").strip('"')
load_env()
print("Credentials successfully loaded from .env!")

Credentials successfully loaded from .env!


In [ ]:
# Popular Nifty 50 tickers mapping to Zerodha Instrument Tokens
SYMBOL_TO_TOKEN = {
    'RELIANCE': '738561', 'TCS': '2953217', 'INFY': '408065', 'HDFCBANK': '341249',
    'ICICIBANK': '1270529', 'SBIN': '779521', 'BHARTIARTL': '2714625', 'ITC': '424961',
    'LT': '2939649', 'HINDUNILVR': '340481', 'AXISBANK': '1510401', 'ASIANPAINT': '6401',
    'M&M': '519937', 'TATASTEEL': '895745', 'WIPRO': '969473', 'TATAMOTORS': '884737',
    'TECHM': '3465729', 'TITAN': '897537', 'ULTRACEMCO': '2920961', 'COALINDIA': '5219585',
    'JSWSTEEL': '3001857', 'NTPC': '2977281', 'POWERGRID': '3834113', 'SUNPHARMA': '857857',
    'ADANIENT': '6405', 'ADANIPORTS': '3861249', 'APOLLOHOSP': '40193', 'BAJAJ-AUTO': '4265217',
    'BAJAJFINSV': '4269569', 'BAJFINANCE': '81153', 'BPCL': '134657', 'BRITANNIA': '141057',
    'CIPLA': '177665', 'DIVISLAB': '2800641', 'DRREDDY': '225537', 'EICHERMOT': '134657', 'GRASIM': '315393',
    'HCLTECH': '1850625', 'HDFCLIFE': '3696897', 'HEROMOTOCO': '345089', 'HINDALCO': '348929',
    'INDUSINDBK': '1346049', 'KOTAKBANK': '492033', 'LTIM': '4658433', 'NESTLEIND': '4544769',
    'ONGC': '633601', 'SBILIFE': '5633', 'TATACONSUM': '878337', 'WIPRO': '969473', 'TRENT': '502017'
}
# Filter out mapping errors
SYMBOL_TO_TOKEN = {k: v for k, v in SYMBOL_TO_TOKEN.items() if isinstance(v, str) and v.isdigit()}
print(f"Configured {len(SYMBOL_TO_TOKEN)} Nifty 50 stocks for backtesting.")

SyntaxError: invalid syntax (2296689727.py, line 11)

In [ ]:
def fetch_candles(symbol, token):
    api_key = os.environ.get('ZERODHA_API_KEY')
    access_token = os.environ.get('ZERODHA_ACCESS_TOKEN')
    if not api_key or not access_token:
        raise ValueError("Missing Zerodha API Key or Access Token")
        
    # Fetch 5 years of daily data
    to_date = pd.Timestamp.now().strftime('%Y-%m-%d')
    from_date = (pd.Timestamp.now() - pd.DateOffset(years=5)).strftime('%Y-%m-%d')
    
    url = f"https://api.kite.trade/instruments/historical/{token}/day"
    headers = {
        'X-Kite-Version': '3',
        'Authorization': f'token {api_key}:{access_token}',
        'User-Agent': 'Mozilla/5.0'
    }
    params = {'from': from_date, 'to': to_date}
    
    resp = requests.get(url, headers=headers, params=params, timeout=15)
    if resp.status_code == 200:
        data = resp.json()
        if 'data' in data and 'candles' in data['data']:
            candles = data['data']['candles']
            df = pd.DataFrame(candles, columns=['date', 'open', 'high', 'low', 'close', 'volume'])
            df['date'] = pd.to_datetime(df['date'])
            return df
    return None
print("Fetch helper successfully defined!")

Fetch helper successfully defined!


In [ ]:
def calculate_indicators(df):
    # 200-day Simple Moving Average (SMA)
    df['sma200'] = df['close'].rolling(200).mean()
    
    # MACD Calculation
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema12 - ema26
    df['signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['hist'] = df['macd'] - df['signal']
    
    # 3-day Return % (to verify price change build up)
    df['pct_change_3d'] = df['close'].pct_change(3) * 100
    return df
print("Indicators helper successfully defined!")

Indicators helper successfully defined!


In [ ]:
def backtest_macd_strategy(df):
    if df is None or len(df) < 250:
        return {'total_trades': 0, 'win_rate': 0.0, 'avg_pnl': 0.0, 'trades': []}
        
    trades = []
    in_trade = False
    buy_price = 0
    buy_date = None
    hold_days = 0
    
    for i in range(200, len(df) - 1):
        row = df.iloc[i]
        prev_row = df.iloc[i - 1]
        
        # Upward Crossover Condition
        macd_crossover_up = (prev_row['macd'] <= prev_row['signal']) and (row['macd'] > row['signal'])
        # Downward Crossover Condition
        macd_crossover_down = (prev_row['macd'] >= prev_row['signal']) and (row['macd'] < row['signal'])
        
        # Filter Rules
        price_above_sma200 = row['close'] > row['sma200']
        price_building_up = row['pct_change_3d'] > 0
        
        if not in_trade:
            # Buy trigger: MACD crossover + Price filter + Momentum filter
            if macd_crossover_up and price_above_sma200 and price_building_up:
                next_open = df.iloc[i + 1]['open']
                buy_price = next_open
                buy_date = df.iloc[i + 1]['date']
                in_trade = True
                hold_days = 0
        else:
            hold_days += 1
            # Exit trigger: MACD downward crossover OR 20-day time hold limit
            if macd_crossover_down or hold_days >= 20:
                next_open = df.iloc[i + 1]['open']
                sell_price = next_open
                sell_date = df.iloc[i + 1]['date']
                pnl = ((sell_price - buy_price) / buy_price) * 100
                trades.append({
                    'entry_date': buy_date,
                    'entry_price': buy_price,
                    'exit_date': sell_date,
                    'exit_price': sell_price,
                    'pnl': pnl
                })
                in_trade = False
                
    # Force close any open trade at the end
    if in_trade:
        sell_price = df.iloc[-1]['close']
        sell_date = df.iloc[-1]['date']
        pnl = ((sell_price - buy_price) / buy_price) * 100
        trades.append({
            'entry_date': buy_date,
            'entry_price': buy_price,
            'exit_date': sell_date,
            'exit_price': sell_price,
            'pnl': pnl
        })
        
    total_trades = len(trades)
    wins = [t for t in trades if t['pnl'] > 0]
    win_rate = (len(wins) / total_trades * 100) if total_trades > 0 else 0.0
    avg_pnl = (sum(t['pnl'] for t in trades) / total_trades) if total_trades > 0 else 0.0
    
    return {
        'total_trades': total_trades,
        'win_rate': win_rate,
        'avg_pnl': avg_pnl,
        'trades': trades
    }
print("Backtester helper successfully defined!")

Backtester helper successfully defined!


In [ ]:
results = []
print("Starting backtest execution across Nifty 50 stocks (Zerodha API data)...")
for sym, token in SYMBOL_TO_TOKEN.items():
    try:
        df = fetch_candles(sym, token)
        if df is not None and len(df) > 250:
            df = calculate_indicators(df)
            res = backtest_macd_strategy(df)
            results.append({
                'symbol': sym,
                'trades_count': res['total_trades'],
                'win_rate': res['win_rate'],
                'avg_pnl': res['avg_pnl']
            })
            print(f"✔ {sym}: Trades: {res['total_trades']}, Win Rate: {res['win_rate']:.1f}%, Avg PnL: {res['avg_pnl']:.2f}%")
        else:
            print(f"✖ {sym}: Insufficient historical data")
    except Exception as e:
        print(f"✖ {sym}: Failed to backtest ({str(e)})")

df_results = pd.DataFrame(results)
print("\nBacktesting completed successfully!")

[OK] ADANIENT: Trades: 14, Win Rate: 35.7%, Avg PnL: 0.52%
[OK] APOLLOHOSP: Trades: 34, Win Rate: 41.2%, Avg PnL: -0.06%
[OK] ASIANPAINT: Trades: 19, Win Rate: 31.6%, Avg PnL: 0.33%
[OK] BAJFINANCE: Trades: 26, Win Rate: 26.9%, Avg PnL: -1.50%
[OK] HDFCLIFE: Trades: 21, Win Rate: 28.6%, Avg PnL: -0.65%
[OK] BPCL: Trades: 26, Win Rate: 42.3%, Avg PnL: 0.57%
[OK] BRITANNIA: Trades: 29, Win Rate: 44.8%, Avg PnL: 0.34%
[OK] CIPLA: Trades: 29, Win Rate: 27.6%, Avg PnL: -0.06%
[OK] DRREDDY: Trades: 27, Win Rate: 37.0%, Avg PnL: -0.06%
[OK] EICHERMOT: Trades: 34, Win Rate: 44.1%, Avg PnL: 1.11%
[OK] GRASIM: Trades: 34, Win Rate: 32.4%, Avg PnL: -0.16%
[OK] HDFCBANK: Trades: 26, Win Rate: 38.5%, Avg PnL: 0.34%
[OK] HEROMOTOCO: Trades: 29, Win Rate: 44.8%, Avg PnL: 1.13%
[OK] HINDALCO: Trades: 23, Win Rate: 39.1%, Avg PnL: -0.00%
[OK] HINDUNILVR: Trades: 16, Win Rate: 31.2%, Avg PnL: -0.67%
[OK] INFY: Trades: 13, Win Rate: 30.8%, Avg PnL: -1.19%
[OK] ITC: Trades: 23, Win Rate: 30.4%, Avg PnL: 0

In [ ]:
if not df_results.empty:
    # Summary Performance Report
    print("====================================================")
    print("           MACD Crossover Summary Report            ")
    print("====================================================")
    print(f"Total Stocks Backtested  : {len(df_results)}")
    print(f"Average Win Rate (%)      : {df_results['win_rate'].mean():.2f}%")
    print(f"Average Return per Trade  : {df_results['avg_pnl'].mean():.2f}%")
    print(f"Total Trades Generated    : {df_results['trades_count'].sum()}")
    print("====================================================")
    
    print("\nTop 5 Performing Stocks by Win Rate:")
    display(df_results.sort_values(by='win_rate', ascending=False).head(5))
else:
    print("No results to display.")

           MACD Crossover Summary Report            
Total Stocks Backtested  : 48
Average Win Rate (%)      : 37.87%
Average Return per Trade  : 0.21%
Total Trades Generated    : 1276

Top 5 Performing Stocks by Win Rate:


Symbol,Trades Count,Win Rate (%),Avg PnL (%)
BAJAJ-AUTO,29,58.6%,1.91%
TITAN,28,50.0%,0.91%
POWERGRID,22,50.0%,0.71%
TECHM,20,50.0%,0.47%
M&M,33,48.5%,0.85%
